In [20]:
 #Question-1- Loading a Large CSV
#Dataset. activity_log_raw.csv
 from google.colab import drive
 drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:

import pandas as pd
file_name = '/content/drive/MyDrive/DATASETS/raw/Copy of activity_log_raw.csv'

In [7]:
#cleaning the dataset
import pandas as pd

def clean_data_csv(file_name, output):
    """Cleans a CSV file by removing null bytes and handling decoding errors."""
    with open(file_name, "r", encoding="ISO-8859-1", errors="ignore") as infile, \
         open(output, "w", encoding="ISO-8859-1") as outfile:
        for line in infile:
            cleaned = line.replace("\0", " ")
            outfile.write(cleaned)  # cleaning the CSV file

# File paths
file_name = '/content/drive/MyDrive/DATASETS/raw/Copy of activity_log_raw.csv'
cleaned_file = "cleaned_Copy_of_activity_log_raw.csv"

# Call the function
clean_data_csv(file_name, cleaned_file)


In [26]:
data = pd.read_csv(
    cleaned_file,
    encoding="ISO-8859-1",
    on_bad_lines='skip',
    chunksize=100000
)

# Iterate through the chunks and print the head of the first chunk only
for chunk in data:
    print(chunk.head())  # Print the first 5 rows of the first chunk
    break  # Stop after the first chunk




   SID  ACTIVITY_ID                    ACTIVITY_TIME STATUS
0  584         1291  13-APR-15 10.33.42.190000000 PM      S
1  584         1286  13-APR-15 10.33.42.190000000 PM      S
2  584         1285  13-APR-15 10.33.42.190000000 PM      S
3  584         1284  13-APR-15 10.33.42.190000000 PM      S
4  584         1288  13-APR-15 10.33.42.190000000 PM      S


In [10]:
#Task-1. Write a function which returns the date range in the dataset (i.e, earliest and oldest).
#Please call your function and print out the date range.
import pandas as pd

def get_date_range(file_name):
    """
    Returns the earliest and latest dates in the dataset based on the ACTIVITY_TIME column.
    """
    cleaned_file = "cleaned_" + file_name.split('/')[-1]

    # Clean the CSV file first
    clean_data_csv(file_name, cleaned_file)

    # Load the cleaned file in chunks
    data = pd.read_csv(
        cleaned_file,
        encoding="ISO-8859-1",
        on_bad_lines='skip',
        chunksize=100000
    )

    # Extract min and max dates from the ACTIVITY_TIME column
    min_date = None
    max_date = None

    for chunk in data:
        # Convert ACTIVITY_TIME to datetime
        chunk['ACTIVITY_TIME'] = pd.to_datetime(
            chunk['ACTIVITY_TIME'],
            format='%d-%b-%y %I.%M.%S.%f %p',
            errors='coerce'
        )

        # Drop rows where the conversion failed
        chunk = chunk.dropna(subset=['ACTIVITY_TIME'])

        # Update min and max dates
        current_min = chunk['ACTIVITY_TIME'].min()
        current_max = chunk['ACTIVITY_TIME'].max()

        if min_date is None or current_min < min_date:
            min_date = current_min
        if max_date is None or current_max > max_date:
            max_date = current_max

    return min_date, max_date

# File path
file_name = '/content/drive/MyDrive/DATASETS/raw/Copy of activity_log_raw.csv'

# Call the function and print the result
earliest_date, latest_date = get_date_range(file_name)
print(f"Earliest Date: {earliest_date}")
print(f"Latest Date: {latest_date}")


Earliest Date: 2012-08-15 20:01:36.621000
Latest Date: 2015-04-13 22:39:00.431000


In [11]:
#Task-2. Write a function which returns the year and month  with the largest number of events.
import pandas as pd

def get_largest_event_month(file_name):
    """
    Returns the year and month with the largest number of events based on the ACTIVITY_TIME column.
    """
    cleaned_file = "cleaned_" + file_name.split('/')[-1]

    # Clean the CSV file first
    clean_data_csv(file_name, cleaned_file)

    # Load the cleaned file in chunks
    data = pd.read_csv(
        cleaned_file,
        encoding="ISO-8859-1",
        on_bad_lines='skip',
        chunksize=100000
    )

    # Dictionary to store event counts by year-month
    year_month_counts = {}

    for chunk in data:
        # Convert ACTIVITY_TIME to datetime
        chunk['ACTIVITY_TIME'] = pd.to_datetime(
            chunk['ACTIVITY_TIME'],
            format='%d-%b-%y %I.%M.%S.%f %p',
            errors='coerce'
        )

        # Drop rows where the conversion failed
        chunk = chunk.dropna(subset=['ACTIVITY_TIME'])

        # Extract year and month
        chunk['YearMonth'] = chunk['ACTIVITY_TIME'].dt.to_period('M')

        # Count events per year-month
        counts = chunk['YearMonth'].value_counts()

        # Update the global counts
        for year_month, count in counts.items():
            if year_month in year_month_counts:
                year_month_counts[year_month] += count
            else:
                year_month_counts[year_month] = count

    # Find the year and month with the maximum events
    largest_year_month = max(year_month_counts, key=year_month_counts.get)
    max_events = year_month_counts[largest_year_month]

    return str(largest_year_month), max_events

# File path
file_name = '/content/drive/MyDrive/DATASETS/raw/Copy of activity_log_raw.csv'

# Call the function and print the result
largest_year_month, max_events = get_largest_event_month(file_name)
print(f"Year and Month with the largest number of events: {largest_year_month}")
print(f"Number of Events: {max_events}")


Year and Month with the largest number of events: 2013-05
Number of Events: 5689349


In [14]:
#Question-2- Standard Error of the Mean (SEM) with Bootstrapping
#Loading the Dataset. hh_ml.zip
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!unzip '/content/drive/MyDrive/DATASETS/raw/hh_data_ml.zip' -d '/content'

Archive:  /content/drive/MyDrive/DATASETS/raw/hh_data_ml.zip
  inflating: /content/hh_data_ml.csv  

In [16]:
df = pd.read_csv('/content/hh_data_ml.csv', sep = '|', nrows=200000)



<ipython-input-16-03a40a7ac8c1>:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/hh_data_ml.csv', sep = '|', nrows=200000)


In [24]:
df.head()
print(df.shape)

(200000, 27)


In [18]:
pip install dask

In [23]:
#Task-1. Calculate the SEM for age in the dataset. Please use no less than 100 bootstrap samples for your calculation.
#Write a function which takes innumber of  bootstrap samples and returns the SEM.
#Key columns in the dataset. P07M - birth month; P07A- birth year
import pandas as pd
import numpy as np
import dask.dataframe as dd

def read_data_dask(file_path):
    """Reads the data using Dask, specifying dtype for problematic columns."""
    ddf = dd.read_csv(file_path, sep='|', dtype={'P08': 'object'})
    return ddf

def calculate_age(ddf):
    """Calculates age from birth year and month."""
    ddf['age'] = 2025 - ddf['P07A'] - (ddf['P07M'] / 12)
    return ddf

def calculate_bootstrap_means(age_data, num_bootstrap_samples):
    """Performs bootstrapping and calculates bootstrap means."""
    bootstrap_means = []
    for _ in range(num_bootstrap_samples):
        bootstrap_sample = np.random.choice(age_data, size=len(age_data), replace=True)
        bootstrap_means.append(np.mean(bootstrap_sample))
    return bootstrap_means

def calculate_age_sem_dask(file_path, num_bootstrap_samples=100):
    """Calculates the Standard Error of the Mean (SEM) for age using bootstrapping and Dask."""
    ddf = read_data_dask(file_path)
    ddf = calculate_age(ddf)

    # Compute the age data
    age_data = ddf['age'].dropna().compute()

    # Calculate bootstrap means
    bootstrap_means = calculate_bootstrap_means(age_data, num_bootstrap_samples)

    # Calculate SEM as the standard deviation of the bootstrap means
    sem = np.std(bootstrap_means)
    return sem

# File path to your dataset - update this to the actual file path
file_path = '/content/hh_data_ml.csv'  # Replace this with the actual path to your dataset

# Calculate SEM using Dask, 100 bootstrap samples
sem_age = calculate_age_sem_dask(file_path, num_bootstrap_samples=100)

# Print SEM result
print(f"SEM for age: {sem_age}")



SEM for age: 0.5027375470473793


In [54]:
#Question-3- Weather Forecast for All Capital Cities in Africa
#Loading the Dataset Open Weather API; Africa_Cities.csv
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
file_path = '/content/drive/My Drive/Copy of Africa_Cities.csv'
africa_cities = pd.read_csv(file_path)
print(africa_cities.head())
print(africa_cities.shape)


   OBJECTID  Join_Count               CITY_NAME GMI_ADMIN  ADMIN_NAME  \
0         1           1               Jamestown       SHN  St. Helena   
1         2           1                   Praia       CPV  Cape Verde   
2         3           1  Santa Cruz de Tenerife   ESP-CNR    Canarias   
3         4           1              Las Palmas   ESP-CNR    Canarias   
4         5           1                 Funchal   PRT-MDR     Madeira   

  FIPS_CNTRY  CNTRY_NAME                           STATUS     POP  POP_RANK  \
0         SH  St. Helena  National and provincial capital     637         7   
1         CV  Cape Verde                 National capital  113364         5   
2         SP       Spain               Provincial capital  222190         5   
3         SP       Spain                            Other  378495         4   
4         PO    Portugal               Provincial capital  100847         5   

            POP_CLASS  PORT_ID  LABEL_FLAG CONTINENT          SQMI  \
0    Less than 5

In [22]:
#Task-1. Generate a CSV file with weather forecast for Monday, January 13, 2025 as provided by open weather API for all African national capital cities.
import requests
import csv
import datetime
import time
import random

# Inserting My API key
api_key = '0615e5de462b9eb2298bf13c785631cf'

# Complete list of African national capitals
city_data = [
    {"name": "Algiers, Algeria", "continent": "Africa"},
    {"name": "Luanda, Angola", "continent": "Africa"},
    {"name": "Porto-Novo, Benin", "continent": "Africa"},
    {"name": "Gaborone, Botswana", "continent": "Africa"},
    {"name": "Ouagadougou, Burkina Faso", "continent": "Africa"},
    {"name": "Bujumbura, Burundi", "continent": "Africa"},
    {"name": "Praia, Cape Verde", "continent": "Africa"},
    {"name": "Yaoundé, Cameroon", "continent": "Africa"},
    {"name": "Bangui, Central African Republic", "continent": "Africa"},
    {"name": "N'Djamena, Chad", "continent": "Africa"},
    {"name": "Moroni, Comoros", "continent": "Africa"},
    {"name": "Brazzaville, Congo", "continent": "Africa"},
    {"name": "Kinshasa, Democratic Republic of the Congo", "continent": "Africa"},
    {"name": "Djibouti, Djibouti", "continent": "Africa"},
    {"name": "Cairo, Egypt", "continent": "Africa"},
    {"name": "Malabo, Equatorial Guinea", "continent": "Africa"},
    {"name": "Asmara, Eritrea", "continent": "Africa"},
    {"name": "Addis Ababa, Ethiopia", "continent": "Africa"},
    {"name": "Libreville, Gabon", "continent": "Africa"},
    {"name": "Banjul, Gambia", "continent": "Africa"},
    {"name": "Accra, Ghana", "continent": "Africa"},
    {"name": "Conakry, Guinea", "continent": "Africa"},
    {"name": "Bissau, Guinea-Bissau", "continent": "Africa"},
    {"name": "Nairobi, Kenya", "continent": "Africa"},
    {"name": "Maseru, Lesotho", "continent": "Africa"},
    {"name": "Monrovia, Liberia", "continent": "Africa"},
    {"name": "Tripoli, Libya", "continent": "Africa"},
    {"name": "Antananarivo, Madagascar", "continent": "Africa"},
    {"name": "Lilongwe, Malawi", "continent": "Africa"},
    {"name": "Bamako, Mali", "continent": "Africa"},
    {"name": "Nouakchott, Mauritania", "continent": "Africa"},
    {"name": "Port Louis, Mauritius", "continent": "Africa"},
    {"name": "Rabat, Morocco", "continent": "Africa"},
    {"name": "Maputo, Mozambique", "continent": "Africa"},
    {"name": "Windhoek, Namibia", "continent": "Africa"},
    {"name": "Niamey, Niger", "continent": "Africa"},
    {"name": "Abuja, Nigeria", "continent": "Africa"},
    {"name": "Kigali, Rwanda", "continent": "Africa"},
    {"name": "São Tomé, São Tomé and Príncipe", "continent": "Africa"},
    {"name": "Dakar, Senegal", "continent": "Africa"},
    {"name": "Victoria, Seychelles", "continent": "Africa"},
    {"name": "Freetown, Sierra Leone", "continent": "Africa"},
    {"name": "Mogadishu, Somalia", "continent": "Africa"},
    {"name": "Pretoria, South Africa", "continent": "Africa"},
    {"name": "Juba, South Sudan", "continent": "Africa"},
    {"name": "Khartoum, Sudan", "continent": "Africa"},
    {"name": "Mbabane, Eswatini", "continent": "Africa"},
    {"name": "Dodoma, Tanzania", "continent": "Africa"},
    {"name": "Lomé, Togo", "continent": "Africa"},
    {"name": "Tunis, Tunisia", "continent": "Africa"},
    {"name": "Kampala, Uganda", "continent": "Africa"},
    {"name": "Lusaka, Zambia", "continent": "Africa"},
    {"name": "Harare, Zimbabwe", "continent": "Africa"}
]

# Function to fetch weather data from the OpenWeather API
def fetch_weather(city_name):
    try:
        url = f"http://api.openweathermap.org/data/2.5/forecast?q={city_name}&appid={api_key}"
        response = requests.get(url)
        data = response.json()

        if data['cod'] != "200":
            print(f"Error fetching weather for {city_name}: {data.get('message', 'Unknown error')}")
            return []

        return data['list']
    except Exception as e:
        print(f"Error fetching weather for {city_name}: {e}")
        return []

# Function to filter forecasts for a specific date
def filter_forecast_by_date(forecasts, target_date):
    return [
        {
            "datetime": forecast["dt_txt"],
            "temp": forecast["main"]["temp"] - 273.15,  # Convert temperature to Celsius
            "temp_min": forecast["main"]["temp_min"] - 273.15,  # Convert temperature to Celsius
            "temp_max": forecast["main"]["temp_max"] - 273.15,  # Convert temperature to Celsius
            "humidity": forecast["main"]["humidity"],
            "weather_main": forecast["weather"][0]["main"],
            "clouds": forecast["clouds"]["all"]
        }
        for forecast in forecasts if target_date in forecast["dt_txt"]
    ]

# Shuffle city_data to randomize the order of cities
random.shuffle(city_data)

# Fetch and save weather data for African capitals
def fetch_and_save_weather_balanced():
    target_date = "2025-01-13"  # Forecast date
    weather_data = []

    # Tracking how many forecasts per city to limit dominance
    max_forecasts_per_city = 3

    for city in city_data:
        if city["continent"] == "Africa":
            city_name = city["name"]
            forecasts = fetch_weather(city_name)
            filtered_forecasts = filter_forecast_by_date(forecasts, target_date)

            # Limit the number of forecasts per city
            limited_forecasts = filtered_forecasts[:max_forecasts_per_city]

            for forecast in limited_forecasts:
                # Split city into country and city
                country, city = city_name.split(", ")
                weather_data.append([
                    country,
                    city,
                    forecast["datetime"].split(" ")[0],  # Date
                    forecast["datetime"].split(" ")[1],  # Time
                    forecast["weather_main"],
                    forecast["temp"],
                    forecast["temp_min"],
                    forecast["temp_max"],
                    forecast["humidity"],
                    forecast["clouds"]
                ])

            time.sleep(1)  # To respect API rate limits

    # Save data to CSV
    with open("weather_forecast_jan13_2025_balanced.csv", "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Country", "City", "Date", "Time", "Weather_main", "Temp", "Temp_min", "Temp_max", "humidity", "Clouds"])
        writer.writerows(weather_data)

    print("Balanced weather forecast saved to weather_forecast_jan13_2025_balanced.csv")

# Runing the script
fetch_and_save_weather_balanced()


Balanced weather forecast saved to weather_forecast_jan13_2025_balanced.csv


In [69]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('weather_forecast_jan13_2025_balanced.csv')

# Display the first 5 rows
print(df.head())
print(df.tail())

   Country      City        Date      Time Weather_main   Temp  Temp_min  \
0   Kigali    Rwanda  2025-01-13  00:00:00       Clouds  15.73     15.73   
1   Kigali    Rwanda  2025-01-13  03:00:00       Clouds  14.85     14.85   
2   Kigali    Rwanda  2025-01-13  06:00:00       Clouds  20.10     20.10   
3  Yaoundé  Cameroon  2025-01-13  00:00:00       Clouds  19.86     19.86   
4  Yaoundé  Cameroon  2025-01-13  03:00:00       Clouds  21.31     21.31   

   Temp_max  humidity  Clouds  
0     15.73        87      98  
1     14.85        89      39  
2     20.10        68      39  
3     19.86        94      50  
4     21.31        82      67  
        Country        City        Date      Time Weather_main   Temp  \
154  Porto-Novo       Benin  2025-01-13  03:00:00       Clouds  23.23   
155  Porto-Novo       Benin  2025-01-13  06:00:00       Clouds  22.32   
156       Praia  Cape Verde  2025-01-13  00:00:00        Clear  22.64   
157       Praia  Cape Verde  2025-01-13  03:00:00        Cl